In [2]:
import os
from huggingface_hub import hf_hub_download
from gliner import GLiNER

In [3]:
repo_id = "urchade/gliner_base" # urchade/gliner_medium-v2, urchade/gliner_medium-v2.1, urchade/gliner_medium-v2
filenames = ["gliner_config.json", "pytorch_model.bin"]
destination = "search_models"

os.makedirs(destination, exist_ok=True)

for file in filenames:
    hf_hub_download(
        repo_id=repo_id,
        filename=file,
        local_dir=destination
    )

gliner_config.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/792M [00:00<?, ?B/s]

In [4]:
extractor = GLiNER.from_pretrained(destination, load_tokenizer=True)
extractor

UniEncoderSpanGLiNER(
  (model): UniEncoderSpanModel(
    (token_rep_layer): Encoder(
      (bert_layer): Transformer(
        (model): DebertaV2Model(
          (embeddings): DebertaV2Embeddings(
            (word_embeddings): Embedding(128004, 768, padding_idx=0)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): DebertaV2Encoder(
            (layer): ModuleList(
              (0-11): 12 x DebertaV2Layer(
                (attention): DebertaV2Attention(
                  (self): DisentangledSelfAttention(
                    (query_proj): Linear(in_features=768, out_features=768, bias=True)
                    (key_proj): Linear(in_features=768, out_features=768, bias=True)
                    (value_proj): Linear(in_features=768, out_features=768, bias=True)
                    (pos_dropout): Dropout(p=0.1, inplace=False)
                    (dropout): Dropout

In [5]:
extractor.model

UniEncoderSpanModel(
  (token_rep_layer): Encoder(
    (bert_layer): Transformer(
      (model): DebertaV2Model(
        (embeddings): DebertaV2Embeddings(
          (word_embeddings): Embedding(128004, 768, padding_idx=0)
          (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): DebertaV2Encoder(
          (layer): ModuleList(
            (0-11): 12 x DebertaV2Layer(
              (attention): DebertaV2Attention(
                (self): DisentangledSelfAttention(
                  (query_proj): Linear(in_features=768, out_features=768, bias=True)
                  (key_proj): Linear(in_features=768, out_features=768, bias=True)
                  (value_proj): Linear(in_features=768, out_features=768, bias=True)
                  (pos_dropout): Dropout(p=0.1, inplace=False)
                  (dropout): Dropout(p=0.1, inplace=False)
                )
                (output): De

In [6]:
text = """
Cristiano Ronaldo dos Santos Aveiro (Portuguese pronunciation: [kɾiʃˈtjɐnu ʁɔˈnaldu]; born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both Saudi Pro League club Al Nassr and the Portugal national team. Widely regarded as one of the greatest players of all time, Ronaldo has won five Ballon d'Or awards,[note 3] a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes, the most by a European player. He has won 33 trophies in his career, including seven league titles, five UEFA Champions Leagues, the UEFA European Championship and the UEFA Nations League. Ronaldo holds the records for most appearances (183), goals (140) and assists (42) in the Champions League, goals in the European Championship (14), international goals (128) and international appearances (205). He is one of the few players to have made over 1,200 professional career appearances, the most by an outfield player, and has scored over 850 official senior career goals for club and country, making him the top goalscorer of all time.
"""

labels = ["person", "award", "date", "competitions", "teams"]

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Cristiano Ronaldo dos Santos Aveiro => person
5 February 1985 => date
Al Nassr => teams
Portugal national team => teams
Ballon d'Or => award
UEFA Men's Player of the Year Awards => award
European Golden Shoes => award
UEFA Champions Leagues => competitions
UEFA European Championship => competitions
UEFA Nations League => competitions
Champions League => competitions
European Championship => competitions


In [7]:
labels = ["Author", "Book_Title_Only", "Genre", "Keywords"]

In [8]:
text = "Brandon Sanderson series with a girl with long hair that takes place in Space"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Brandon Sanderson => Author
Space => Genre


In [9]:
text = "book where there was a sword on the cover, but it looked kind of old and had oldish lettering. It had characters with powers too"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

characters => Keywords
powers => Keywords


In [10]:
text = "That one Defensive Baking series, I think by Kingfisher something"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Defensive Baking => Book_Title_Only
Kingfisher => Author


In [11]:
text = "Fantasy book with dragons in it by Brandom Mull I think"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Fantasy => Genre
dragons => Keywords
Brandom Mull => Author


In [12]:
text = "A Wizard's Guide To Defensive Baking"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

A Wizard's Guide To Defensive Baking => Book_Title_Only


In [13]:
text = "there were dragons in it and I think it was by Brandom Mull"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

dragons => Genre
Brandom Mull => Author


In [14]:
text = "Garth Nix's Keys to the Kingdom series"

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Garth Nix => Author
Keys to the Kingdom => Book_Title_Only


In [15]:
text = (
    "Looking for a Stephen King book. "
    "I don’t remember a lot about it. It was about this white male authour who lives in Maine."
)

entities = extractor.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

Stephen King => Author


In [17]:
import torch

save_dir = os.path.join(destination, "onnx")
file_name = "glinner.onnx"

os.makedirs(save_dir, exist_ok=True)

# input_tensor = torch.ones((2, 3, 224, 224), dtype=torch.float32)

extractor.export_to_onnx(save_dir=save_dir,
                         onnx_filename=file_name,
                         #quantized_filename="gliner_quantized.onnx",
                         #quantize=True
                         )

# torch.onnx.export(extractor.encoder,
#                 (input_tensor),
#                 output_path,
#                 input_names = ['images'],
#                 output_names = ['embeddings'],
#                 dynamic_shapes=({0: torch.export.Dim.DYNAMIC},),
#                 external_data=False
#                 )

[W530 13:24:45.338931950 shape_type_inference.cpp:1994] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W530 13:24:45.338943504 shape_type_inference.cpp:1994] Warning: The shape inference of prim::PackPadded type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)
[W530 13:24:45.349577306 shape_type_inference.cpp:1994] Warning: The shape inference of prim::PadPacked type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (function UpdateReliable)


{'onnx_path': 'search_models/onnx/glinner.onnx', 'quantized_path': None}

In [18]:
import os

os.path.join(save_dir, file_name)

'search_models/onnx/glinner.onnx'

In [19]:
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process

quant_pre_process(os.path.join(save_dir, file_name),
                os.path.join(save_dir, "gliner_quant_pre.onnx"),
                skip_optimization=False,
                skip_symbolic_shape=True,
                verbose=3)

quantize_dynamic(os.path.join(save_dir, "gliner_quant_pre.onnx"),
                os.path.join(save_dir, "gliner_quant.onnx"),
                weight_type=QuantType.QUInt8)

  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__800"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__801"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__815"
    }
  }
}
.


In [20]:
text = "ONNX is an open-source format designed to enable the interoperability of AI models across various frameworks and tools."
labels = ['format', 'model', 'tool', 'cat']

inputs = extractor._build_dummy_batch(labels, text)
inputs.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'words_mask', 'span_idx', 'span_mask', 'text_lengths'])

In [21]:
spec = extractor._get_onnx_input_spec()
spec

{'input_names': ['input_ids',
  'attention_mask',
  'words_mask',
  'text_lengths',
  'span_idx',
  'span_mask'],
 'output_names': ['logits'],
 'dynamic_axes': {'input_ids': {0: 'batch_size', 1: 'sequence_length'},
  'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
  'words_mask': {0: 'batch_size', 1: 'sequence_length'},
  'text_lengths': {0: 'batch_size', 1: 'value'},
  'span_idx': {0: 'batch_size', 1: 'num_spans', 2: 'idx'},
  'span_mask': {0: 'batch_size', 1: 'num_spans'},
  'logits': {0: 'batch_size',
   1: 'sequence_length',
   2: 'num_spans',
   3: 'num_classes'}}}

In [22]:
all_inputs =  (inputs['input_ids'], inputs['attention_mask'],
                    inputs['words_mask'], inputs['text_lengths'],
                    inputs['span_idx'], inputs['span_mask'])
#all_inputs = dict(zip(spec['input_names'], inputs))
#all_inputs = tuple(inputs[name] for name in spec["input_names"])
len(all_inputs)

6

In [23]:
#wrapper = extractor._create_onnx_wrapper(extractor.model.to("cpu").eval())
class UniEncoderSpanWrapper(torch.nn.Module):
    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(
        self,
        input_ids,
        attention_mask,
        words_mask,
        text_lengths,
        span_idx,
        span_mask,
    ):
        out = self.core(
            input_ids=input_ids,
            attention_mask=attention_mask,
            words_mask=words_mask,
            text_lengths=text_lengths,
            span_idx=span_idx,
            span_mask=span_mask,
        )
        return out.logits if hasattr(out, "logits") else out[0]

wrapper = UniEncoderSpanWrapper(extractor.model.eval())

In [24]:
inputs['words_mask']

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  2,  3,  4,  0,  0,  5,
          6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,  0]])

In [25]:
import torch

save_dir = os.path.join(destination, "onnx")
file_name = "glinner.onnx"

os.makedirs(save_dir, exist_ok=True)

# input_tensor = torch.ones((2, 3, 224, 224), dtype=torch.float32)

# extractor.export_to_onnx(save_dir=save_dir,
#                          onnx_filename=file_name,
#                          quantized_filename="gliner_quantized.onnx",
#                          quantize=True
#                          )
wrapper.eval()
torch.onnx.export(wrapper,
                all_inputs,
                os.path.join(save_dir, file_name),
                input_names = spec['input_names'],
                output_names = spec['output_names'],
                dynamic_axes= spec['dynamic_axes'],
                dynamo=False,
                external_data=False
                )

/home/kojo/tmp/ipykernel_22187/4162465896.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrapper,
/home/kojo/Code/ByItsCover/bic-explore/benv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
[W530 13:28:37.732011378 shape_type_inference.cpp:1994] Warning: The shape inference of prim::P

In [26]:
extractor.config.to_json_file(os.path.join(save_dir, "gliner_config.json"))

extractor.data_processor.transformer_tokenizer.save_pretrained(save_dir)

('search_models/onnx/tokenizer_config.json',
 'search_models/onnx/tokenizer.json')

In [27]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
import onnx

save_dir = os.path.join(destination, "onnx")

quant_pre_process(os.path.join(save_dir, file_name),
                os.path.join(save_dir, "gliner_quant_pre.onnx"),
                skip_optimization=False,
                skip_symbolic_shape=True,
                verbose=3)

model_onnx = onnx.load(os.path.join(save_dir, "gliner_quant_pre.onnx"))

nodes_to_exclude = set()
for node in model_onnx.graph.node:
    # Add the node name
    nodes_to_exclude.add(node.name)

# Create base attention layer paths
attention_paths = [
    "/output/dense",
    "/intermediate/dense"
]

# Generate the full paths for all encoder layers
attention_layer_paths = [
    f"/token_rep_layer/bert_layer/model/encoder/layer.{layer_num}{path}"
    for layer_num in range(12)  # 12 encoder layers (0-11)
    for path in attention_paths
]

# Update nodes_to_exclude
nodes_to_exclude_v2 = [
    node for node in nodes_to_exclude 
    if any(node.startswith(prefix) for prefix in attention_layer_paths)
]

quantize_dynamic(os.path.join(save_dir, "gliner_quant_pre.onnx"),
                os.path.join(save_dir, "gliner_quant.onnx"),
                nodes_to_exclude=nodes_to_exclude_v2,
                per_channel=True,
                weight_type=QuantType.QUInt8)

  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__800"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__801"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 2
    }
    dim {
      dim_param: "unk__815"
    }
  }
}
.
